In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.1.2/spark-3.1.2-bin-hadoop2.7.tgz
!tar xf spark-3.1.2-bin-hadoop2.7.tgz
!pip install pyspark==3.2.1 py4j==0.10.9.3
!pip install -q findspark
!wget https://repo1.maven.org/maven2/com/nvidia/rapids-4-spark_2.12/21.12.0/rapids-4-spark_2.12-21.12.0.jar
!wget https://repo1.maven.org/maven2/ai/rapids/cudf/21.12.2/cudf-21.12.2-cuda11.jar
import os
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ['SPARK_HOME'] = "/kaggle/working/spark-3.1.2-bin-hadoop2.7"
os.environ['PYSPARK_SUBMIT_ARGS'] = "--jars /kaggle/working/rapids-4-spark_2.12-21.12.0.jar,/kaggle/working/cudf-21.12.2-cuda11.jar --master local[*] pyspark-shell"
import findspark
findspark.init()

*Notebook created by Gwydyon Marchelli, Alessandro Ciocchetti, Olga Cozzolino, Davide Bottinelli, Lorena Volpini during the II level Master Big Data Analytics and Social Mining, University of Pisa*

In [ ]:
!nvidia-smi

In [ ]:
from pyspark.sql import SQLContext, SparkSession
from pyspark.conf import SparkConf

spark = SparkSession.builder.appName('SparkRAPIDS').config('spark.plugins','com.nvidia.spark.SQLPlugin').getOrCreate()
spark.sparkContext.addPyFile('/kaggle/working/rapids-4-spark_2.12-21.12.0.jar')
spark.sparkContext.addPyFile('/kaggle/working/cudf-21.12.2-cuda11.jar')
spark.conf.set('spark.rapids.sql.enabled','true')
spark.conf.set('spark.rapids.sql.incompatibleOps.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.read.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.enabled', 'true')

In [ ]:
from pyspark.ml.linalg import DenseMatrix, Vectors
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import MinMaxScaler, VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier as RFC
from pyspark.mllib.linalg import SparseVector
from pyspark.mllib.regression import LabeledPoint
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from pyspark.mllib.evaluation import MulticlassMetrics


In [ ]:
%matplotlib inline
from random import random
import math
from pyspark import SparkContext
import numpy as np
import re
import matplotlib.pyplot as plt
from pyspark.sql import functions
import pandas as pd
import seaborn as sns

## Data Description

In [ ]:
df_games = spark.read.format("csv").load("../input/chess/games.csv", header='true', inferSchema='true', multiline = True)

In [ ]:
df_games.printSchema()

In [ ]:
summary = df_games.describe().toPandas()

In [ ]:
summary

In [ ]:
df_games.show(15)

In [ ]:
#no missing values
from pyspark.sql.functions import isnan, when, count, col, isnull, concat_ws, split
missing = df_games.select([count(when(isnull(c), c)).alias(c) for c in df_games.columns]).show()

In [ ]:
#we joined the two columns to create a new target variable
df_games = df_games.withColumn("Target", concat_ws("-",'winner','victory_status'))

In [ ]:
df_games.show(5)

In [ ]:
df = df_games.withColumn('fix_time', split(df_games.increment_code, "\\+")[0]).withColumn('variable_time', split(df_games['increment_code'], '\\+')[1])
df.show(5)

In [ ]:
#dropping no needed columns. created at and last move at dropped for dataset inconsistencies
toDrop = ('created_at', 'last_move_at', 'victory_status', 'increment_code', 'moves')
df = df.drop(*toDrop)

In [ ]:
#casting variables
df = df.withColumn("fix_time",df.fix_time.cast('int'))
df = df.withColumn("variable_time",df.variable_time.cast('int'))
df = df.where(df.Target != "draw-outoftime") #we don't want records of target draw out of time beacuse we can consider them outliers compared with draws fix_time

In [ ]:
df.dtypes

In [ ]:
#we dropped columns with not-continuus variables, to create the correlation vector
df_corr = df.drop('id', 'rated', 'winner', 'white_id', 'black_id', 'opening_eco', 'opening_name', 'Target')
features = df_corr.schema.names

vectorassembler = VectorAssembler(inputCols = features, outputCol= 'assemblerfeatures')

output_dataset = vectorassembler.transform(df_corr)  
pearsonCorr = Correlation.corr(output_dataset, 'assemblerfeatures', 'pearson').collect()[0][0]

#trasformiamo la DenseMatrix in un array numpy
correlation_array = pearsonCorr.toArray()

correlationDF = pd.DataFrame(
    correlation_array,
    index = features,
    columns = features
)
correlationDF

In [ ]:
#heatmap
sns.set(rc={'figure.figsize':(11.7,8.27)})
sns.heatmap(correlationDF, annot=True)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
ax=sns.histplot(data=df.toPandas(), x="winner", color='black')
plt.xlabel('Winner', fontsize=16);
plt.ylabel('Count', fontsize=16);

In [ ]:
ax = sns.histplot(data=df.toPandas(), x="Target",color='black')
ax.tick_params(axis='x', rotation=45)

In [ ]:
sns.violinplot(data=df.toPandas(), x="opening_ply", y="white_rating")

In [ ]:
sns.violinplot(data=df.toPandas(), x="opening_ply", y="black_rating")

In [ ]:
ax=sns.boxplot(data=df.toPandas(), x="Target", y="fix_time",color = 'black', medianprops=dict(color='red', alpha=1),showmeans=True,
            meanprops={"marker":"o",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"10"}, showfliers = True) 
ax.tick_params(axis='x', rotation=45)
ax.set(ylim=(-10, 170))

In [ ]:
ax=sns.boxplot(data=df.toPandas(), x="Target", y="variable_time",color = 'black', medianprops=dict(color='white', alpha=1),showmeans=True,
            meanprops={"marker":"o",
                       "markerfacecolor":"white", 
                       "markeredgecolor":"black",
                      "markersize":"10"}, showfliers = False) 
ax.tick_params(axis='x', rotation=45)
ax.set(ylim=(-5, 30))

In [ ]:
b = df.toPandas()['opening_eco'].value_counts(normalize=True)
b= b*100
ax = b.plot.bar(x=[], y=b,color='black', edgecolor='black') 
ax.tick_params(axis='x', rotation=90)
plt.xlabel('Opening Code', fontsize=16)
plt.ylabel('Frequency (%)', fontsize=16)

ax.axes.xaxis.set_ticklabels([])



# Classification

In [ ]:
# split the dataset into training and test set to get ready for random forest classification
# first case "Target" is the target variable

to_drop_class = ("id", "white_id", "black_id", "opening_name", "winner")
df_rf1 = df.drop(*to_drop_class)

#indexing of categorical features
inputs = ["Target", "opening_eco"]
outputs = ["Target_", "opening_eco_"]
indexer = StringIndexer(inputCols=inputs, outputCols=outputs, handleInvalid="skip")

df_index = indexer.fit(df_rf1).transform(df_rf1)


df_index.show(50)

In [ ]:
#features vector creation
num_col = ["rated","turns", "white_rating", "black_rating", "opening_ply", "fix_time", "variable_time", "opening_eco_"]
assembler = VectorAssembler(inputCols=num_col, outputCol="features")

output_dataset = assembler.transform(df_index)

classificationData = output_dataset.select("features", "Target_")

train = classificationData.sampleBy("Target_", fractions={0: 0.7, 1: 0.7, 2: 0.7, 3: 0.7, 4: 0.7, 5: 0.7, 6: 0.7 })

test = classificationData.subtract(train)

In [ ]:
rf = RFC(featuresCol='features', labelCol='Target_', impurity='gini')

eval_accuracy = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="Target_", metricName="accuracy")
eval_precision = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="Target_", metricName="precisionByLabel")
eval_recall = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="Target_", metricName="recallByLabel")
eval_f1 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="Target_", metricName="f1")


In [ ]:
# Grid Search for hyperparameter tuning
rfparamGrid = (ParamGridBuilder()

               .addGrid(rf.maxDepth, [5,15])  

               .addGrid(rf.maxBins, [365])

               .addGrid(rf.numTrees, [5, 15, 30, 60])  
             .build())
              
#best: 0.38 di accuracy con 15 max depth, 365 max bins, 60 trees.

In [ ]:
# Create 5-fold CrossValidator
rfcv = CrossValidator(estimator = rf,
                      estimatorParamMaps = rfparamGrid,
                      evaluator = eval_accuracy,
                      numFolds = 5)

# Run cross validations.
rfcvModel = rfcv.fit(train)


# Use test set here so we can measure the accuracy of our model on new data
rfpredictions = rfcvModel.transform(test)


In [ ]:
rfcvModel.getEstimatorParamMaps()[ np.argmax(rfcvModel.avgMetrics) ]

In [ ]:
print(
    f"Overall Accuracy: {eval_accuracy.evaluate(rfpredictions)} \n Overall Precision: {eval_precision.evaluate(rfpredictions)} \n Overall Recall: {eval_recall.evaluate(rfpredictions)} \n  Overall F1: {eval_f1.evaluate(rfpredictions)}" 
)

In [ ]:
rfpredictions.show(5)

In [ ]:
preds_and_labels = rfpredictions.select(['prediction','Target_']).withColumn('label', F.col('Target_').cast(FloatType())).orderBy('prediction')

#select only prediction and label columns
preds_and_labels = preds_and_labels.select(['prediction','label'])

metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

c7_confusionMx = metrics.confusionMatrix().toArray()

MulticlassLabels = ['white_resign','black_resign','white_mate', 'black_mate', 'draw-draw', 'black_outoftime','white_outoftime']

In [ ]:
c7_confusionMx

In [ ]:
sommaMatrix=np.sum(c7_confusionMx, axis=1)

In [ ]:
axes = sns.heatmap(c7_confusionMx/sommaMatrix[:,None], annot=True, fmt='.2%') 
axes.set_title('Confusion Matrix');
axes.set_xlabel('\nPredicted Values')
axes.set_ylabel('Actual Values ');
axes.set_xticklabels(['white_resign','black_resign','white_mate', 'black_mate', 'draw-draw', 'black_outoftime','white_outoftime'], rotation=45)
axes.set_yticklabels(['white_resign','black_resign','white_mate', 'black_mate', 'draw-draw', 'black_outoftime','white_outoftime'], rotation=0)
plt.savefig('C7_ConfMatrix_HPSA_chess.png',dpi=300)

In [ ]:
diagonal_matrix = np.array([1.38, 49.34, 14.61, 4.65, 59.4,  24.22,   1.96])
labelEsito = np.array(['white-outoftime','black-resign','white-mate','draw-draw','white-resign','black-mate','black-outoftime'])

In [ ]:
ax = sns.histplot(data=df.toPandas(), x="Target",color='black', stat='percent',zorder=1)  #nelle y mettere in

ax= sns.scatterplot(labelEsito,diagonal_matrix,s=300, color="red", marker="*",alpha=1, zorder=2)
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.set(ylim=(0, 65))
ax.set(xlim=(-0.5, 6.5))

In [ ]:
# CASE  2

In [ ]:
# split the dataset into training and test set 
# case 2)  target is "winner"
to_drop_class = ("id", "white_id", "black_id", "opening_name", "Target")
df_rf_case2 = df.drop(*to_drop_class)
inputs_case2 = ["winner", "opening_eco"]
outputs_case2 = ["winner_", "opening_eco_"]
indexer_case2 = StringIndexer(inputCols=inputs_case2, outputCols=outputs_case2, handleInvalid="skip")

df_index_case2 = indexer_case2.fit(df_rf_case2).transform(df_rf_case2)
df_index_case2 = df_index_case2.drop("opening_eco", "winner")

df_index_case2.show(50)

In [ ]:

num_col_case2 = ["rated","turns", "white_rating", "black_rating", "opening_ply", "fix_time", "variable_time", "opening_eco_"]
assembler_case2 = VectorAssembler(inputCols=num_col_case2, outputCol="features")

output_dataset_case2 = assembler_case2.transform(df_index_case2)

classificationData_case2 = output_dataset_case2.select("features", "winner_")

train_case2 = classificationData_case2.sampleBy("winner_", fractions={0: 0.7, 1: 0.7, 2: 0.7})

test_case2 = classificationData_case2.subtract(train_case2)

In [ ]:
rf_case2 = RFC(featuresCol='features', labelCol='winner_', impurity='gini')

eval_accuracy_case2 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="winner_", metricName="accuracy") 
eval_precision_case2  = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="winner_", metricName="precisionByLabel")
eval_recall_case2 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="winner_", metricName="recallByLabel")
eval_f1_case2 = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="winner_", metricName="f1")



In [ ]:
# Grid Search for hyperparameter tuning
rfparamGrid_case2 = (ParamGridBuilder()

               .addGrid(rf_case2.maxDepth, [5, 15])

               .addGrid(rf_case2.maxBins, [365])

               .addGrid(rf.numTrees, [5, 15, 30, 60])
             .build())

# Create 5-fold CrossValidator
rfcv_case2 = CrossValidator(estimator = rf_case2,
                      estimatorParamMaps = rfparamGrid_case2,
                      evaluator = eval_accuracy_case2,
                      numFolds = 5)

# Run cross validations.
rfcvModel_case2 = rfcv_case2.fit(train_case2)


# Use test set here so we can measure the accuracy of our model on new data
rfpredictions_case2 = rfcvModel_case2.transform(test_case2)



In [ ]:
rfcvModel_case2.getEstimatorParamMaps()[ np.argmax(rfcvModel.avgMetrics) ]

In [ ]:
# cvModel uses the best model found from the Cross Validation
# Evaluate best model
print('Overall Accuracy:', eval_accuracy_case2.evaluate(rfpredictions_case2))
print('Overall Precision:', eval_precision_case2.evaluate(rfpredictions_case2)) 
print('Overall Recall:', eval_recall_case2.evaluate(rfpredictions_case2)) 
print('Overall F1:', eval_f1_case2.evaluate(rfpredictions_case2)) 

In [ ]:
preds_and_labels2 = rfpredictions_case2.select(['prediction','winner_']).withColumn('label', F.col('winner_').cast(FloatType())).orderBy('prediction')

#select only prediction and label columns
preds_and_labels2 = preds_and_labels2.select(['prediction','label'])

metrics = MulticlassMetrics(preds_and_labels2.rdd.map(tuple))

c3_confusionMx = metrics.confusionMatrix().toArray()

MulticlassLabels2 = ['white','black', 'draw']

In [ ]:
sommaMatrix2=np.sum(c3_confusionMx, axis=1)
c3_confusionMx

In [ ]:
axes = sns.heatmap(c3_confusionMx/sommaMatrix2[:,None], annot=True, fmt='.2%') 
axes.set_title('Confusion Matrix');
axes.set_xlabel('\nPredicted Values')
axes.set_ylabel('Actual Values ');
axes.set_xticklabels(MulticlassLabels2, rotation=0)
axes.set_yticklabels(MulticlassLabels2, rotation=0)
plt.savefig('C7_ConfMatrix_HPSA_chess.png',dpi=300)

In [ ]:
diagonal_matrix2 = np.array([77.19, 52.31,  0])
labelEsito2 = np.array(['white','black','draw'])

In [ ]:
ax=sns.histplot(data=df.toPandas(), x="winner", color='black',stat='percent',zorder=1) 
ax= sns.scatterplot(labelEsito2,diagonal_matrix2,s=300, color="red", marker="*",alpha=1, zorder=2)


ax.set(ylim=(0, 95))
ax.set(xlim=(-0.5, 2.5))